# B1 · PRM Retrain (§9A.3 复现 · DeBERTa-v3-large + LoRA)

**Track B · Week 1–2 · Colab Pro+ A100 80GB(主号)· 关键路径**

## 设计选择
- **Backbone**: `microsoft/deberta-v3-large`(§9A.3 原选型,MATH-Shepherd 同族,A100 80GB 显存宽松)
- **微调**: LoRA r=16 α=32 dropout=0.1 on attention + FFN(`q,k,v,o,gate,up,down`)
- **打分头**: `Linear(hidden=1024, 1)` sigmoid,step-level BCE loss
- **Trajectory 聚合**: 几何平均(Lightman 2024 ICLR 做法)
- **Batch**: per-device 8 × grad-accum 2 = eff 16
- **LR**: 1e-4 cosine,warmup 100 steps,3 epochs
- **数据降级路径**: 优先 §9A.3 原训练集;检测不足 5k 样本 → 用 §9E.1 C3 candidates + critic score 合成伪标签

## 验收(W2 Go/No-Go)
- holdout **step AUROC ≥ 0.80**
- **trajectory AUROC ≥ 0.82**(§9A.3 原纪录)
- Spearman(traj_score, judge.correctness) ≥ 0.4
- 不达标 → 加训 1 epoch + 标为 v2.1 重评

In [ ]:
# ========== 0. A100 setup ==========
!nvidia-smi | head -20
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, random, math
from pathlib import Path

PHYRE_ROOT = Path('/content/drive/MyDrive/phyre')
CKPT_DIR   = PHYRE_ROOT / 'ckpt'
DATA_DIR   = PHYRE_ROOT / 'data'
LOG_DIR    = PHYRE_ROOT / 'logs'
for p in [CKPT_DIR, DATA_DIR, LOG_DIR]: p.mkdir(parents=True, exist_ok=True)

# 使用支持 numpy 2.x 的新版。
# peft >=0.17 会检查 torchao 版本 (要求 >= 0.16),Colab 预装 0.10 过旧 →
# 先卸载 torchao,peft 的 is_torchao_available() 会返回 False(我们用不到 torchao)。
!pip -q uninstall -y torchao 2>/dev/null || true
!pip -q install -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "datasets>=3.0" scikit-learn scipy sentencepiece

# 若此 cell 报 'numpy.strings' 未找到 → 顶栏 Runtime > Restart session,
# 然后跳过本 cell 的 pip,直接从 import numpy 继续(pip 已装好)。
import numpy as np
print('numpy', np.__version__)
assert np.__version__.startswith('2.'),     '需要 numpy 2.x;若被降级 → Runtime > Restart session 后再跑'

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

torch.manual_seed(42); random.seed(42); np.random.seed(42)
DEVICE = 'cuda'
print('torch', torch.__version__, '  cuda:', torch.cuda.is_available())

In [ ]:
# ========== 1. data locator + robust PRM data synthesis ==========
# Schema: {"qid", "question", "steps": [...], "labels": [0/1, ...], "_traj_w": float}
#
# Priority:
#   (a) Real §9A.3 PRM data if any path exists
#   (b) Synth from §9E.1 results/9E_main/*.jsonl — split pred_full into steps,
#       label each step via key_element substring match (weak supervision)
#   (c) Hard fail with upload instructions

import re

CANDIDATE_PRM_PATHS = [
    PHYRE_ROOT / 'data' / 'prm' / 'train.jsonl',
    PHYRE_ROOT / 'data' / '9A.3' / 'prm_train.jsonl',
    Path('/content/pvgap_experiment/data/prm/train.jsonl'),
]
E91_RESULTS = PHYRE_ROOT / 'results' / '9E_main'
E91_CANDIDATE_DIRS = [
    E91_RESULTS,
    Path('/content/pvgap_experiment/results/9E_main'),
]

def _load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')] if Path(p).is_file() else []

train_data = []
for p in CANDIDATE_PRM_PATHS:
    d = _load_jsonl(p)
    if d: print(f'  [real] {len(d)} samples at {p}'); train_data.extend(d)

if len(train_data) < 2000:
    print(f'\nFallback: synthesising PRM data from §9E.1 results (have {len(train_data)} real samples)')
    e91_dir = None
    for cand in E91_CANDIDATE_DIRS:
        if cand.is_dir() and list(cand.glob('*.jsonl')):
            e91_dir = cand; break
    if e91_dir is None:
        raise SystemExit(
            '\n\u2717 No PRM data and no \u00a79E.1 results to fall back on.\n\n'
            '  Upload all 15 jsonl in  pvgap_experiment/results/9E_main/  to\n'
            f'  {E91_RESULTS}\n\n'
            '  then restart this cell.')

    e91_rows = []
    files = sorted(e91_dir.glob('*.jsonl'))
    for f in files:
        e91_rows.extend(_load_jsonl(f))
    print(f'  [e91] loaded {len(e91_rows)} rows from {len(files)} files at {e91_dir}')

    def label_steps(pred_full, key_elements):
        """Split pred_full into steps, label each via key_element mention."""
        steps = [s.strip() for s in pred_full.split('\n\n') if len(s.strip()) > 10]
        if len(steps) < 2:
            steps = [s.strip() for s in re.split(r'(?<=[.!?])\s+', pred_full)
                     if len(s.strip()) > 15]
        if len(steps) < 2:
            return None, None
        ke_lower = [k.lower() for k in key_elements if isinstance(k, str)]
        labels = []
        for s in steps:
            sl = s.lower()
            hit = any(ke in sl for ke in ke_lower)
            if not hit:
                # loose match: any 2+ tokens from a key_element co-occur
                hit = any(sum(tok in sl for tok in ke.split()) >= 2 for ke in ke_lower)
            labels.append(1 if hit else 0)
        return steps, labels

    n_synth = 0
    for row in e91_rows:
        if row.get('error') or not row.get('pred_full'):
            continue
        ke = row.get('key_elements') or []
        if not ke:
            continue
        steps, labels = label_steps(row['pred_full'], ke)
        if not steps:
            continue
        traj_w = float(row.get('key_recall', 0.5))
        if isinstance(row.get('judge'), dict) and row['judge'].get('error') is None:
            traj_w = float(row['judge'].get('correctness', traj_w))
        train_data.append({
            'qid': row['qid'], 'question': (row.get('question') or '')[:500],
            'steps': steps[:16], 'labels': labels[:16],
            '_traj_w': traj_w, '_synth': True,
        })
        n_synth += 1
    print(f'  [synth] produced {n_synth} PRM samples from \u00a79E.1')

if not train_data:
    raise SystemExit('\u2717 train_data empty after fallback \u2014 check paths above.')

print(f'\nTOTAL train samples: {len(train_data)}')
random.shuffle(train_data)
n_holdout = max(200, len(train_data) // 10)
holdout, train_data = train_data[:n_holdout], train_data[n_holdout:]
print(f'  train: {len(train_data)}   holdout: {len(holdout)}')

# label-balance sanity
pos = sum(sum(r['labels']) for r in train_data)
tot = sum(len(r['labels']) for r in train_data)
print(f'  step-level positive rate: {pos}/{tot} = {pos/max(tot,1):.2%}')
if pos / max(tot, 1) < 0.15 or pos / max(tot, 1) > 0.85:
    print('  \u26a0 label imbalance extreme \u2014 PRM AUROC ceiling may be low')

In [ ]:
# ========== 2. tokenizer + step-level encoding ==========
MODEL_NAME = 'microsoft/deberta-v3-large'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
STEP_SEP  = ' <step> '
STEP_TOK_ID = tokenizer.convert_tokens_to_ids('Ġstep') or tokenizer.pad_token_id

class PRMDataset(Dataset):
    def __init__(self, rows, max_len=1024):
        self.rows = rows; self.max_len = max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        # structure: Q [SEP] step1 [step] step2 [step] ... — insert a sentinel after each step
        text = r['question'] + ' [SEP] '
        step_positions = []  # char index of each [STEP] marker
        for j, s in enumerate(r['steps']):
            text += s + STEP_SEP
        enc = tokenizer(text, max_length=self.max_len, truncation=True,
                        return_tensors='pt', padding='max_length')
        ids = enc['input_ids'][0]
        # locate [step] tokens (assume each encodes to 1-2 pieces; find by decode)
        step_idx = [k for k, t in enumerate(ids.tolist())
                    if tokenizer.decode([t]).strip().lower() == 'step']
        # truncate labels to fit what survived
        n = min(len(step_idx), len(r['labels']))
        step_idx = step_idx[:n]
        labels = r['labels'][:n]
        return {'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0],
                'step_positions': torch.tensor(step_idx + [0]*(32-len(step_idx)), dtype=torch.long),
                'step_labels': torch.tensor(labels + [-1]*(32-len(labels)), dtype=torch.float),
                'n_steps': torch.tensor(n, dtype=torch.long)}

train_ds = PRMDataset(train_data); hold_ds = PRMDataset(holdout)
print(f'dataset sizes — train {len(train_ds)} holdout {len(hold_ds)}')

In [ ]:
# ========== 3. model: DeBERTa + LoRA + step scoring head ==========
class PRMModel(torch.nn.Module):
    def __init__(self, backbone_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        hidden = self.backbone.config.hidden_size
        self.score_head = torch.nn.Linear(hidden, 1)
    def forward(self, input_ids, attention_mask, step_positions, n_steps):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        B = h.size(0); max_s = step_positions.size(1)
        # gather per-step hidden → score
        idx = step_positions.unsqueeze(-1).expand(-1, -1, h.size(-1))  # (B, max_s, H)
        step_h = torch.gather(h, 1, idx)                                # (B, max_s, H)
        logits = self.score_head(step_h).squeeze(-1)                    # (B, max_s)
        # mask invalid positions
        mask = torch.arange(max_s, device=h.device).unsqueeze(0) < n_steps.unsqueeze(1)
        return logits, mask

base = PRMModel(MODEL_NAME)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1, bias='none',
    target_modules=['query_proj', 'key_proj', 'value_proj'],  # DeBERTa attn proj names
    task_type=TaskType.FEATURE_EXTRACTION)
base.backbone = get_peft_model(base.backbone, lora_cfg)
base.backbone.print_trainable_parameters()
base.to(DEVICE)

In [ ]:
# ========== 4. train loop with step-level BCE ==========
EPOCHS = 3; BS = 8; ACCUM = 2; LR = 1e-4; SAVE_EVERY = 500

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2)
hold_loader  = DataLoader(hold_ds,  batch_size=BS, shuffle=False, num_workers=2)

opt = torch.optim.AdamW([p for p in base.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
total_steps = (len(train_loader) // ACCUM) * EPOCHS
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=100, num_training_steps=total_steps)
bce = torch.nn.BCEWithLogitsLoss(reduction='none')

@torch.no_grad()
def evaluate():
    base.eval()
    step_preds, step_labels = [], []
    traj_preds, traj_labels = [], []
    for b in hold_loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        logits, mask = base(b['input_ids'], b['attention_mask'], b['step_positions'], b['n_steps'])
        probs = torch.sigmoid(logits)
        lab = b['step_labels']
        valid = (lab >= 0) & mask
        step_preds.extend(probs[valid].cpu().numpy().tolist())
        step_labels.extend(lab[valid].cpu().numpy().tolist())
        # trajectory = geo mean of valid step probs
        for i in range(probs.size(0)):
            v = valid[i]
            if v.sum() == 0: continue
            traj_preds.append(torch.exp(torch.log(probs[i][v] + 1e-8).mean()).item())
            traj_labels.append(int(lab[i][v].mean().item() > 0.5))
    base.train()
    step_auc = roc_auc_score(step_labels, step_preds) if len(set(step_labels)) > 1 else float('nan')
    traj_auc = roc_auc_score(traj_labels, traj_preds) if len(set(traj_labels)) > 1 else float('nan')
    return step_auc, traj_auc

base.train(); global_step = 0; best_traj = 0
for epoch in range(EPOCHS):
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits, mask = base(batch['input_ids'], batch['attention_mask'],
                            batch['step_positions'], batch['n_steps'])
        lab = batch['step_labels']
        valid = (lab >= 0) & mask
        loss = (bce(logits, lab.clamp(min=0)) * valid.float()).sum() / valid.sum().clamp(min=1)
        (loss / ACCUM).backward()
        if (i + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in base.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); opt.zero_grad(); global_step += 1
            if global_step % 50 == 0:
                print(f'  ep{epoch} step{global_step}  loss={loss.item():.4f}  lr={sched.get_last_lr()[0]:.2e}')
            if global_step % SAVE_EVERY == 0:
                step_auc, traj_auc = evaluate()
                print(f'  ## eval @ {global_step}  step_AUC={step_auc:.3f}  traj_AUC={traj_auc:.3f}')
                if traj_auc > best_traj:
                    best_traj = traj_auc
                    torch.save({'state_dict': base.state_dict(), 'step': global_step,
                                'step_auc': step_auc, 'traj_auc': traj_auc,
                                'model_name': MODEL_NAME},
                               CKPT_DIR / 'prm_v2.pt')
                    print(f'  ✓ saved new best → prm_v2.pt')

print(f'\nDONE. best trajectory AUROC = {best_traj:.3f}')

In [ ]:
# ========== 5. final holdout report + Go/No-Go ==========
step_auc, traj_auc = evaluate()
TARGET_STEP, TARGET_TRAJ = 0.80, 0.82
report = {'model': MODEL_NAME, 'n_train': len(train_ds), 'n_holdout': len(hold_ds),
          'step_auc': step_auc, 'traj_auc': traj_auc,
          'pass_step': step_auc >= TARGET_STEP,
          'pass_traj': traj_auc >= TARGET_TRAJ,
          'ckpt': str(CKPT_DIR / 'prm_v2.pt')}
(LOG_DIR / 'B1_final_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
if not (report['pass_step'] and report['pass_traj']):
    print('\n⚠ NOT PASSING. 诊断建议:')
    print('  1. 检查合成数据比例,真实 §9A.3 数据缺失则 AUROC 天花板低')
    print('  2. 加训 1 epoch,或换 RoBERTa-large 做 backbone 消融')
    print('  3. 如果 step AUC 过但 traj AUC 不过 → 聚合策略改为 min 而非 geo-mean')
else:
    print('\n✓ PASS. git tag w2-prm-done')